<a href="https://colab.research.google.com/github/RaihahMahmud/FlyRank-AI--starter-ML-Internship-/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


The model output is converted into a ranked review queue for content editors.

Pages with higher model scores are placed earlier in the queue because they more closely match the observed click-volume pattern used to train the model.

Each page also receives a reason code based on observed search-performance signals:

- `CTR_OPPORTUNITY` — relatively high impressions with low observed CTR.
- `HIGH_IMPRESSIONS` — high observed impressions and therefore a larger visibility opportunity to review.
- `LOW_CTR` — low observed CTR relative to the training-data threshold.
- `MONITOR` — no strong signal for immediate prioritization.

The queue is intended to prioritize **human review**. A high rank does not mean that a page should automatically be refreshed.

# Section 1 code — Ranked action queue

In [9]:

queue = base[
    [
        "priority_rank",
        "client_hash_id",
        "content_hash_id",
        "model_score",
        "reason_code",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
    ]
].copy()

# Keep the highest-priority pages first
queue = queue.sort_values("priority_rank").reset_index(drop=True)

print("RANKED ACTION QUEUE")
print("-------------------")
print("Total pages:", len(queue))
print("\nTop 20 pages for human review:")

display(queue.head(20))

RANKED ACTION QUEUE
-------------------
Total pages: 176738

Top 20 pages for human review:


,priority_rank,client_hash_id,content_hash_id,model_score,reason_code,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,1.0,MONITOR,331,2,0.008193,14.129210
1,2,client_ff644d8251367cbb,content_fe05b276762746d7,1.0,HIGH_IMPRESSIONS,1575,7,0.006445,3.319639
2,3,client_2094c6eb080311d5,content_6264e8d216508596,1.0,MONITOR,137,3,0.029186,6.943486
3,4,client_2094c6eb080311d5,content_658a48362f36873e,1.0,MONITOR,97,2,0.040323,9.850325
4,5,client_2094c6eb080311d5,content_658e508f2349745e,1.0,MONITOR,212,2,0.005163,8.461884
5,6,client_2094c6eb080311d5,content_660ea1d2d414547d,1.0,MONITOR,61,1,0.008065,13.814881
6,7,client_2094c6eb080311d5,content_66396d27d05fed3b,1.0,MONITOR,294,4,0.007866,5.302337
7,8,client_2094c6eb080311d5,content_66399b01e068e192,1.0,MONITOR,212,2,0.018280,6.819488
8,9,client_2094c6eb080311d5,content_669e5069458769b7,1.0,MONITOR,733,17,0.015215,4.335280
9,10,client_2094c6eb080311d5,content_68496c187a59d72f,1.0,MONITOR,489,6,0.019289,6.131286


# ML-10 — Recreating validated model and generate page-level scores

In [10]:
####code

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = march[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
    ]
].copy()

df = df.dropna(subset=["gsc_impressions", "gsc_clicks"])

df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

base = (
    df.groupby(["client_hash_id", "content_hash_id"], as_index=False)
      .agg({
          "gsc_impressions": "sum",
          "gsc_clicks": "sum",
          "ctr": "mean",
          "gsc_avg_position": "mean"
      })
)

                                                            # Remove rows with missing model features
base = base.dropna(
    subset=["gsc_impressions", "ctr", "gsc_avg_position"]
).copy()

                               # Honest client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(base, groups=base["client_hash_id"])
)

train = base.iloc[train_idx].copy()
test = base.iloc[test_idx].copy()

                                            # Learn all thresholds from training data only
click_threshold = train["gsc_clicks"].median()
imp_threshold = train["gsc_impressions"].quantile(0.75)
ctr_threshold = train["ctr"].quantile(0.25)

train["target"] = (
    train["gsc_clicks"] > click_threshold
).astype(int)

test["target"] = (
    test["gsc_clicks"] > click_threshold
).astype(int)

feature_cols = [
    "gsc_impressions",
    "ctr",
    "gsc_avg_position"
]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(train[feature_cols], train["target"])

                                            # Score all pages
base["model_score"] = model.predict_proba(
    base[feature_cols]
)[:, 1]

                                       # Assign action reason codes
base["reason_code"] = np.select(
    [
        (base["gsc_impressions"] >= imp_threshold) &
        (base["ctr"] <= ctr_threshold),

        base["gsc_impressions"] >= imp_threshold,

        base["ctr"] <= ctr_threshold
    ],
    [
        "CTR_OPPORTUNITY",
        "HIGH_IMPRESSIONS",
        "LOW_CTR"
    ],
    default="MONITOR"
)

                            # Rank pages by model score
base = base.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

base["priority_rank"] = np.arange(1, len(base) + 1)

print("Model and action queue recreated successfully")
print("Pages:", len(base))
print("Train rows:", len(train))
print("Test rows:", len(test))
print(
    "Client overlap:",
    len(set(train["client_hash_id"]) & set(test["client_hash_id"]))
)
print("\nReason codes:")
print(base["reason_code"].value_counts())

Model and action queue recreated successfully
Pages: 176738
Train rows: 138310
Test rows: 38428
Client overlap: 0

Reason codes:
reason_code
LOW_CTR             102903
HIGH_IMPRESSIONS     42156
MONITOR              26681
CTR_OPPORTUNITY       4998
Name: count, dtype: int64


## 2. Intended use and limits

### Intended use

The action queue is intended to help content editors prioritize which pages to review first.

The model score and reason code provide a directional ranking based on observed search-performance signals from the March 2026 data. Higher-ranked pages can be reviewed earlier, while the final decision remains with a human reviewer.

### Limits

The score does not predict the causal effect of refreshing a page. It does not establish that a refresh will increase clicks, impressions, CTR, or search rankings.

The model was validated on held-out clients, but the evaluation is based on observed historical data. Changes in traffic patterns, content types, data quality, or other conditions may reduce its usefulness over time.

The queue should therefore be treated as a **decision-support tool**, not an automatic content-action system.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

=

### Human review rules

The ranked list is meant to help a content or SEO specialist decide which pages to look at first. It is not meant to make the decision for them.

Before taking action, the reviewer should:

- Check if the page is still relevant and useful.
- Look at the page's search intent and current content.
- Check whether the impressions, CTR, and position numbers make sense.
- Consider recent changes or other context that the model does not have.
- Decide whether the page should be refreshed, investigated further, or left as it is.

The model score is only a way to prioritize the review.

### No-go list

The model should not be used to:

- Automatically change or publish content.
- Decide on its own that a page needs a refresh.
- Promise better rankings, traffic, or clicks after a refresh.
- Make recommendations when the underlying data is missing or unreliable.
- Override the content or SEO specialist's judgment.
- Treat the model result as proof that refreshing content causes better performance.

The final decision stays with the human reviewer.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

=

The model should be checked regularly rather than assumed to work the same way over time.

### What to monitor

I would keep an eye on:

- Changes in the distribution of impressions, CTR, and average position.
- Missing or unusual values in the model inputs.
- Precision and recall when new labeled data becomes available.
- Whether the types of pages being ranked have changed significantly.

A change in the input data does not automatically mean the model needs to be retrained. It should first be checked to understand whether the change comes from a real shift in the data or from a data-quality problem.

### When to review or retrain

The model should be reviewed if:

- Input distributions change noticeably.
- Data quality or missing values become a problem.
- Precision or recall drops on newly evaluated data.
- The model's rankings no longer seem useful to content reviewers.

Retraining should only happen after checking the reason for the change and confirming that enough new, reliable data is available.

# Section 4 code — Basic monitoring checks

In [11]:


monitor_cols = [
    "gsc_impressions",
    "ctr",
    "gsc_avg_position"
]

print("MONITORING CHECK")
print("----------------")

for col in monitor_cols:
    print(f"\n{col}")
    print("  Missing:", base[col].isna().sum())
    print("  Median:", round(base[col].median(), 4))
    print("  Mean:", round(base[col].mean(), 4))
    print("  95th percentile:", round(base[col].quantile(0.95), 4))

print("\nModel score summary")
print("  Median:", round(base["model_score"].median(), 4))
print("  Mean:", round(base["model_score"].mean(), 4))
print("  Missing:", base["model_score"].isna().sum())

MONITORING CHECK
----------------

gsc_impressions
  Missing: 0
  Median: 173.0
  Mean: 1587.9867
  95th percentile: 7238.15

ctr
  Missing: 0
  Median: 0.0
  Mean: 0.0021
  95th percentile: 0.0098

gsc_avg_position
  Missing: 0
  Median: 8.5053
  Mean: 15.9993
  95th percentile: 56.5

Model score summary
  Median: 0.0097
  Mean: 0.3886
  Missing: 0


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

=

The ranked action queue was exported as a CSV so it can be used for analysis and review outside the notebook.

A separate JSON file contains the main queue and monitoring metrics for use in the research paper.

The exported queue contains 176,738 pages. The most common reason code is `LOW_CTR`, followed by `HIGH_IMPRESSIONS`, `MONITOR`, and `CTR_OPPORTUNITY`.

The CSV is kept out of Git because it is a generated queue. The metrics JSON can be committed with the notebook.

# Section 5 code — Export ranked queue and paper metrics

In [12]:

import os
import json

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Export ranked review queue
queue_path = "work/outputs/w07_ranked_action_queue.csv"
queue.to_csv(queue_path, index=False)

# Save monitoring/model summary for the paper
metrics = {
    "total_pages_ranked": int(len(queue)),
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "model_score_median": float(base["model_score"].median()),
    "model_score_mean": float(base["model_score"].mean()),
    "model_score_missing": int(base["model_score"].isna().sum()),
    "feature_missing": {
        col: int(base[col].isna().sum())
        for col in monitor_cols
    },
    "client_overlap_train_test": 0
}

with open("work/outputs/w07_action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("EXPORTS CREATED")
print("----------------")
print(queue_path)
print("work/outputs/w07_action_playbook_metrics.json")
print("\nRows exported:", len(queue))

EXPORTS CREATED
----------------
work/outputs/w07_ranked_action_queue.csv
work/outputs/w07_action_playbook_metrics.json

Rows exported: 176738


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.